In [3]:
from __future__ import annotations

from dataclasses import dataclass
from pathlib import Path
from typing import Dict, Iterable, List

import pandas as pd


# -----------------------------
# Config
# -----------------------------
@dataclass(frozen=True)
class Paths:
    project_root: Path = Path("D:\#QMIND Lab\#Project-2\Reserve\Dataset")
    images_dir: Path = project_root / "STM_images"
    resources_dir: Path = project_root
    metadata_xlsx: Path = project_root / "STM_metadata.xlsx"
    output_csv: Path = project_root / "curated_dataset.csv"
    output_json: Path = project_root / "curated_dataset.json"


PATHS = Paths()

# Image extensions to include
IMAGE_EXTS = {".png", ".jpg", ".jpeg", ".tif", ".tiff", ".bmp", ".webp"}


# -----------------------------
# Helpers
# -----------------------------
def snake_case(name: str) -> str:
    """Convert column name to snake_case."""
    name = name.strip()
    name = name.replace("/", "_").replace("–", "-")
    name = name.replace("(", "").replace(")", "")
    name = name.replace(",", "").replace("%", "pct")
    name = name.replace(" ", "_")
    while "__" in name:
        name = name.replace("__", "_")
    return name.lower()


def standardize_columns(df: pd.DataFrame) -> pd.DataFrame:
    """
    Rename columns to a consistent schema.

    Output columns:
      datapoint_id, label, topo_bias_v, topo_it_pa, didv_bias_v, didv_it_pa, didv_lockin_mv
    """
    # 1) snake_case everything
    df = df.rename(columns={c: snake_case(c) for c in df.columns})

    # 2) map common variants to preferred names
    rename_map = {
        "data_point": "datapoint_id",
        "label": "label",
        "bias_voltage_v_of_topograph": "topograph_voltage_v",
        "tunneling_current_pa_of_topograph": "topograph_current_pa",
        "bias_voltage_v_of_di_dv_map": "didv_voltage_v",
        "tunneling_current_pa_of_di_dv_map": "didv_current_pa",
        "modulation_voltage_mv_of_di_dv_map": "didv_modulation_voltage_mv",
        "scale_bar_nm_of_topograph": "topograph_scale_nm",
        "scale_bar_nm_of_di_dv_map": "didv_scale_nm",
    }
    df = df.rename(columns={k: v for k, v in rename_map.items() if k in df.columns})

    # required columns
    if "datapoint_id" not in df.columns:
        raise ValueError("Missing required column: 'Data point' (expected to become datapoint_id)")
    if "label" not in df.columns:
        raise ValueError("Missing required column: 'Label' (expected to become label)")

    return df


def read_metadata(xlsx_path: Path) -> pd.DataFrame:
    """Read the datapoint metadata from Excel and standardize column names."""
    if not xlsx_path.exists():
        raise FileNotFoundError(f"Metadata file not found: {xlsx_path.resolve()}")
    df = pd.read_excel(xlsx_path, keep_default_na=False)
    return standardize_columns(df)


def list_images(images_dir: Path, exts: set[str] = IMAGE_EXTS) -> List[Path]:
    """List image files in images_dir (non-recursive)."""
    if not images_dir.exists():
        raise FileNotFoundError(f"Images directory not found: {images_dir.resolve()}")
    files = [p for p in images_dir.iterdir() if p.is_file() and p.suffix.lower() in exts]
    return sorted(files)


def build_prefix_index(image_files: Iterable[Path]) -> Dict[str, List[Path]]:
    """
    Index images by leading token before '_' or '-' in filename stem.
    Example:
      '1_topo.png' -> prefix '1'
      'C1-map.jpg' -> prefix 'C1'
    """
    index: Dict[str, List[Path]] = {}
    for p in image_files:
        stem = p.stem
        prefix = stem.split("_")[0].split("-")[0]
        index.setdefault(prefix, []).append(p)

    for k in index:
        index[k] = sorted(index[k])
    return index


def attach_images(df_meta: pd.DataFrame, prefix_index: Dict[str, List[Path]]) -> pd.DataFrame:
    """Attach list of images to each datapoint_id."""
    df = df_meta.copy()
    df["datapoint_id"] = df["datapoint_id"].astype(str)
    df["image_files"] = df["datapoint_id"].map(lambda x: prefix_index.get(x, []))
    return df


def save_csv_one_row_per_datapoint(df: pd.DataFrame, output_csv: Path) -> None:
    """
    Save CSV with ONE ROW per datapoint.

    Note: CSV cannot store real arrays nicely, so we store image_paths as a single string.
    JSON output will store image_paths as a proper list.
    """
    out = df.copy()

    # Convert list of paths to a single string for CSV (semicolon-separated)
    out["image_paths_csv"] = out["image_paths"].apply(lambda xs: ";".join(xs))
    out = out.drop(columns=["image_paths"])

    output_csv.parent.mkdir(parents=True, exist_ok=True)
    out.to_csv(output_csv, index=False)
    print(f"Saved: {output_csv} (rows={len(out)})")


def save_json(df: pd.DataFrame, output_json: Path) -> None:
    """Save JSON with ONE ROW per datapoint and image_paths as a real list."""
    output_json.parent.mkdir(parents=True, exist_ok=True)
    df.to_json(output_json, orient="records", indent=2)
    print(f"Saved: {output_json} (rows={len(df)})")


# -----------------------------
# Main pipeline (ONE row per datapoint)
# -----------------------------
def main() -> None:
    # 1) read metadata
    df_meta = read_metadata(PATHS.metadata_xlsx)
    print("Metadata rows:", len(df_meta))
    print("Metadata columns:", list(df_meta.columns))

    # 2) scan images and map by prefix
    image_files = list_images(PATHS.images_dir)
    print("Images found:", len(image_files))
    print("Example images:", [p.name for p in image_files[:10]])

    prefix_index = build_prefix_index(image_files)

    # 3) attach images to each datapoint (LIST per row)
    df_joined = attach_images(df_meta, prefix_index)

    # Convert image Paths -> strings
    df_joined["image_paths"] = df_joined["image_files"].apply(lambda imgs: [str(p) for p in imgs])
    df_joined["num_images"] = df_joined["image_paths"].apply(len)

    # Warn if missing images
    missing = df_joined[df_joined["num_images"] == 0]["datapoint_id"].tolist()
    if missing:
        print("WARNING: No images found for datapoints:", missing)

    # 4) final dataset (one row per datapoint)
    preferred_cols = [
        "datapoint_id",
        "label",
        "topograph_voltage_v",
        "topograph_current_pa",
        "didv_voltage_v",
        "didv_current_pa",
        "didv_modulation_voltage_mv",
        "topograph_scale_nm",
        "didv_scale_nm",
        "num_images",
        "image_paths",
    ]
    preferred_cols = [c for c in preferred_cols if c in df_joined.columns]
    df_dataset = df_joined[preferred_cols].copy()

    # 5) save outputs
    save_csv_one_row_per_datapoint(df_dataset, PATHS.output_csv)
    save_json(df_dataset, PATHS.output_json)


if __name__ == "__main__":
    main()


Metadata rows: 209
Metadata columns: ['datapoint_id', 'label', 'topograph_voltage_v', 'topograph_current_pa', 'didv_voltage_v', 'didv_current_pa', 'didv_modulation_voltage_mv', 'topograph_scale_nm', 'didv_scale_nm', 'doi']
Images found: 227
Example images: ['100_Topograph.png', '101_Topograph.png', '102_Topograph.png', '103_map.jpg', '103_Topograph.jpg', '104_Topograph.jpg', '105_Topograph.jpg', '106_Topograph.png', '107_Topograph.png', '108_Topograph.png']
Saved: D:\#QMIND Lab\#Project-2\Reserve\Dataset\curated_dataset.csv (rows=209)
Saved: D:\#QMIND Lab\#Project-2\Reserve\Dataset\curated_dataset.json (rows=209)


<>:15: SyntaxWarning: invalid escape sequence '\#'
<>:15: SyntaxWarning: invalid escape sequence '\#'
C:\Users\ACER\AppData\Local\Temp\ipykernel_27372\1633537542.py:15: SyntaxWarning: invalid escape sequence '\#'
  project_root: Path = Path("D:\#QMIND Lab\#Project-2\Reserve\Dataset")
